- In traditional retail, Market Basket Analysis looks at individual shopping receipts (e.g., "If someone buys diapers, do they buy beer?"). However, because our data is aggregated at the Department level per week, we are looking for Macro-Affinities: "If Department A has a massive spike in sales, does Department B also spike in sales on the exact same week?"
- We then calculate the Pearson Correlation Coefficient. The algorithm scans down the columns and asks: "Every time Department 1's sales go up, do Department 2's sales also go up? If Department 1 crashes, does Department 2 crash?" It outputs a score from -1.0 (perfect opposites) to 1.0 (perfectly synchronized).
- The Business ROI (Why do we care?)
This is where you bring extreme value to the stakeholders using the Dashboard! Imagine the script discovers that Dept 72 (Electronics) and Dept 8 (Video Games) have a 0.88 correlation score. The Regional Manager can now deploy a "Loss Leader" Strategy: They place a massive, unprofitable discount (Markdown) on Electronics. They lose money on the Electronics sale, but because the departments are deeply linked, foot traffic to Video Games spikes, and they make all their profit back there.

De-seasonalized Residual Correlation (or Partial Correlation).

How it works:
Right now, we are running Pearson Correlation on the Raw Sales. If we want to strip away the Holiday/Seasonal bias, we do this:

Calculate the "Expected Seasonal Baseline": For every department, we calculate what we expect them to sell on any given week based purely on history (e.g., Dept 92 always sells $150k on Week 47).
Calculate the "Residual" (The Surprise): We subtract the Expected Sales from the Actual Sales. If Dept 92 sold $180k, the Residual is +$30k (an unexpected $30k spike).
Run Pearson on the Residuals: Instead of running .corr() on the raw sales, we run .corr() on the Surprise Spikes (Residuals).
Why this defeats the Holiday Trap:
If both Turkeys and TVs spike on Black Friday exactly as much as we expected them to, their Residuals are $0. The correlation matrix completely ignores them!

But, if a store runs a massive, unannounced marketing campaign on TVs in March, and TV sales unexpectedly spike by $40k (Residual = +40k), AND suddenly Video Game sales also unexpectedly spike by $15k (Residual = +15k), the algorithm will catch that! It mathematically proves that when TVs unexpectedly surge, Video Games surge too—true causation isolated from seasonality.

In [4]:
import polars as pl
import pandas as pd
import structlog
import config

# 1. Use Polars to lazily scan the correct Gold Parquet.
# We extract 'sales_last_year' to act as our mathematically proven Expected Seasonal Baseline!
lf = pl.scan_parquet(config.GOLD_DATA_PATH).select(['store', 'date', 'dept', 'weekly_sales', 'sales_last_year'])

# Drop the first 52-weeks of warm-up data where the historical baseline doesn't exist yet
lf = lf.drop_nulls(subset=["sales_last_year"])

# 2. De-Seasonalize the data! (The Surprise Factor)
# Residual = Actual Sales - Expected Seasonal Sales
# We are mathematically isolating the "Surprise Spikes" and stripping away the Holiday bias.
lf = lf.with_columns(
    (pl.col("weekly_sales") - pl.col("sales_last_year")).alias("residual_sales")
)

df_collected = lf.collect()

# Cast 'dept' to string so the pivot creates clean string column names
df_collected = df_collected.with_columns(pl.col('dept').cast(pl.String))

# Cast 'dept' to string so the pivot creates clean string column names
df_collected = df_collected.with_columns(pl.col('dept').cast(pl.String))
print(df_collected.head(), df_collected.shape)

shape: (5, 6)
┌───────┬─────────────────────┬──────┬──────────────┬─────────────────┬────────────────┐
│ store ┆ date                ┆ dept ┆ weekly_sales ┆ sales_last_year ┆ residual_sales │
│ ---   ┆ ---                 ┆ ---  ┆ ---          ┆ ---             ┆ ---            │
│ i32   ┆ datetime[μs]        ┆ str  ┆ f64          ┆ f64             ┆ f64            │
╞═══════╪═════════════════════╪══════╪══════════════╪═════════════════╪════════════════╡
│ 1     ┆ 2011-02-04 00:00:00 ┆ 1    ┆ 21665.76     ┆ 24924.5         ┆ -3258.74       │
│ 1     ┆ 2011-02-04 00:00:00 ┆ 2    ┆ 46829.12     ┆ 50605.27        ┆ -3776.15       │
│ 1     ┆ 2011-02-04 00:00:00 ┆ 3    ┆ 11012.52     ┆ 13740.12        ┆ -2727.6        │
│ 1     ┆ 2011-02-04 00:00:00 ┆ 4    ┆ 35870.49     ┆ 39954.04        ┆ -4083.55       │
│ 1     ┆ 2011-02-04 00:00:00 ┆ 5    ┆ 31280.62     ┆ 32229.38        ┆ -948.76        │
└───────┴─────────────────────┴──────┴──────────────┴─────────────────┴────────────────┘ (303121

In [5]:

# 2. Pivot in Polars (happens in Rust for extreme memory efficiency)
basket = df_collected.pivot(
    values='residual_sales', 
    index=['store', 'date'], 
    on='dept',
    aggregate_function='first'
).fill_null(0)

print(basket.head(), '\n', basket.shape)


shape: (5, 83)
┌───────┬─────────────────────┬──────────┬───────────┬───┬─────┬─────┬─────┬─────┐
│ store ┆ date                ┆ 1        ┆ 2         ┆ … ┆ 39  ┆ 50  ┆ 43  ┆ 65  │
│ ---   ┆ ---                 ┆ ---      ┆ ---       ┆   ┆ --- ┆ --- ┆ --- ┆ --- │
│ i32   ┆ datetime[μs]        ┆ f64      ┆ f64       ┆   ┆ f64 ┆ f64 ┆ f64 ┆ f64 │
╞═══════╪═════════════════════╪══════════╪═══════════╪═══╪═════╪═════╪═════╪═════╡
│ 1     ┆ 2011-02-04 00:00:00 ┆ -3258.74 ┆ -3776.15  ┆ … ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ 0.0 │
│ 2     ┆ 2011-02-04 00:00:00 ┆ -5427.08 ┆ -14591.93 ┆ … ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ 0.0 │
│ 3     ┆ 2011-02-04 00:00:00 ┆ 1068.67  ┆ 180.04    ┆ … ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ 0.0 │
│ 4     ┆ 2011-02-04 00:00:00 ┆ -3946.25 ┆ 2719.9    ┆ … ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ 0.0 │
│ 5     ┆ 2011-02-04 00:00:00 ┆ 495.95   ┆ -484.49   ┆ … ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ 0.0 │
└───────┴─────────────────────┴──────────┴───────────┴───┴─────┴─────┴─────┴─────┘ 
 (4095, 83)


In [6]:

logger.info("Calculating Pearson Affinity Matrix...")
# Polars doesn't have a native 2D correlation matrix function.
# We drop the index columns, then cast the isolated feature columns to Pandas to leverage its highly optimized C-backed .corr()
dept_matrix = basket.drop(['store', 'date']).to_pandas()
print(dept_matrix.head(), dept_matrix.shape)


2026-04-23 15:03:03 [info     ] Calculating Pearson Affinity Matrix...
         1         2        3        4        5        6        7        8  \
0 -3258.74  -3776.15 -2727.60 -4083.55  -948.76 -2424.07 -3844.22  -225.68   
1 -5427.08 -14591.93 -4815.08 -1450.28  4737.02 -1713.74 -6392.12 -8178.52   
2  1068.67    180.04  -550.51   331.21  3132.88   121.17  1198.76   -67.28   
3 -3946.25   2719.90 -1506.14 -3100.25  3590.06 -4315.52 -5275.84  -591.85   
4   495.95   -484.49  -700.44   760.37 -2307.16  -438.85    -3.92  -313.99   

         9       10  ...       94       95        96       97       98   99  \
0  1868.83  2315.42  ...  3797.90  7580.02  28899.26  5632.83   306.39  0.0   
1 -1086.90 -4996.06  ...   266.05  2168.13   4034.90  3362.34 -2154.18  0.0   
2  1887.62  1379.75  ...     0.00  1286.45    845.49  -156.60     0.00  0.0   
3  1767.66  4136.28  ...  5192.78  8598.57   2641.16  1840.08  2250.19  0.0   
4   347.03  1718.26  ...     4.85  3359.04   2911.18   684.47    

In [7]:
correlation_matrix = dept_matrix.corr()

print(correlation_matrix.head(), '\n', correlation_matrix.shape)

          1         2         3         4         5         6         7  \
1  1.000000  0.134176  0.064904  0.130080  0.257771  0.080699  0.460336   
2  0.134176  1.000000  0.295042  0.663117  0.097496  0.083281  0.226757   
3  0.064904  0.295042  1.000000  0.240252  0.069137  0.068397  0.070830   
4  0.130080  0.663117  0.240252  1.000000  0.140555  0.120216  0.247365   
5  0.257771  0.097496  0.069137  0.140555  1.000000  0.418325  0.460949   

          8         9        10  ...        94        95        96        97  \
1  0.127518  0.214582  0.112210  ...  0.125519  0.025506  0.063047  0.039075   
2  0.502437  0.351491  0.266449  ...  0.269090  0.566088  0.124934  0.348620   
3  0.199564  0.081925  0.062809  ...  0.054835  0.136087 -0.008321  0.077017   
4  0.579797  0.335544  0.289829  ...  0.403321  0.666219  0.190115  0.514919   
5  0.196953  0.141275  0.129033  ...  0.055646  0.132219  0.046922  0.108190   

         98        99        39        50        43        65  
1  0

In [8]:
logger.info("Isolating strict associative rules (Correlation > 0.6)...")
pairs = correlation_matrix.unstack().reset_index()
pairs.columns = ['Department_A', 'Department_B', 'Correlation_Score']

print(pairs.head(), pairs.shape)


2026-04-23 15:03:03 [info     ] Isolating strict associative rules (Correlation > 0.6)...
  Department_A Department_B  Correlation_Score
0            1            1           1.000000
1            1            2           0.134176
2            1            3           0.064904
3            1            4           0.130080
4            1            5           0.257771 (6561, 3)


In [10]:

# Convert back to integers for clean logic
pairs['Department_A'] = pairs['Department_A'].astype(int)
pairs['Department_B'] = pairs['Department_B'].astype(int)

# Drop self-correlations AND mirror duplicates (A->B is the same as B->A)
# By strictly enforcing A < B, we eliminate the symmetric redundancy!
pairs = pairs[pairs['Department_A'] < pairs['Department_B']]

# Filter for high-confidence rules
strong_rules = pairs[pairs['Correlation_Score'] > 0.6].sort_values('Correlation_Score', ascending=False)

print(strong_rules.head(), strong_rules.shape)

      Department_A  Department_B  Correlation_Score
16               1            18           0.847433
5496            90            92           0.813664
1813            24            33           0.756250
255              4            13           0.741854
93               2            13           0.734386 (33, 3)
